# M1 — Where does the difficulty confound enter *our own* model?

Mechanistic-interpretability section for **"Hard, Not Wrong."** Turns the statistical confound
critique into a *causal, localized* claim: at which pipeline stage does difficulty become linearly
readable, and does projecting out the length direction reproduce the residualization result?

Pipeline: per-step MiniLM embeddings **H** → bottleneck step-codes **Z** → attention-pooled chain **c**.

Covers (all CPU/T4 — no 7B here; M2 stays on the A100):
- **M1a/b** probe each stage for each confound (ridge R² / logistic acc)
- **M1c** causal: project the length direction out of `c`, re-fit A/B, compare to residualization
- **Attention analysis** of the chain head: does it attend to *long* steps (surface) or *error* steps (signal)?

### Setup — create a Kaggle Dataset with these 3 files (all small), then attach it:
- `step_cache.pt`  (data/step_cache.pt)
- `candidates.jsonl`  (data/processed_pb/candidates.jsonl)
- `sdae_best.pt`  (experiments/results_multiseed/ckpts/frozen_seed0/sdae_best.pt)

Set `INP` below to the attached dataset path (e.g. `/kaggle/input/ridae-m1`).


In [ ]:
INP = "/kaggle/input/ridae-m1"   # <-- set to your attached Kaggle dataset folder
import os, glob
print("files:", os.listdir(INP) if os.path.isdir(INP) else "SET INP CORRECTLY")

In [ ]:
# --- StepSDAE_PRM (inlined from main/sdae_prm.py so the notebook is self-contained) ---
import math, torch, torch.nn as nn, torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d, maxlen=512):
        super().__init__()
        pe = torch.zeros(maxlen, d); pos = torch.arange(maxlen).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe", pe)
    def forward(self, x): return x + self.pe[:x.size(1)].unsqueeze(0)

class StepSDAE_PRM(nn.Module):
    def __init__(self, in_dim=384, d_model=256, nhead=4, nlayers=2, dim_ff=512, dropout=0.1):
        super().__init__()
        self.in_dim=in_dim; self.proj=nn.Linear(in_dim,d_model); self.pos=PositionalEncoding(d_model)
        self.mask_token=nn.Parameter(torch.randn(d_model)*0.02)
        layer=nn.TransformerEncoderLayer(d_model,nhead,dim_ff,dropout,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,nlayers)
        self.decoder=nn.Sequential(nn.Linear(d_model,d_model),nn.GELU(),nn.Linear(d_model,in_dim))
        self.prm_head=nn.Linear(d_model,1); self.attn=nn.Linear(d_model,1); self.chain_head=nn.Linear(d_model,1)
    def encode(self, steps, pad_mask, corrupt_mask=None):
        x=self.proj(steps)
        if corrupt_mask is not None:
            x=torch.where(corrupt_mask.unsqueeze(-1), self.mask_token.view(1,1,-1), x)
        x=self.pos(x); return self.encoder(x, src_key_padding_mask=pad_mask)
    def forward(self, steps, pad_mask, corrupt_mask=None):
        h=self.encode(steps,pad_mask,corrupt_mask); recon=self.decoder(h)
        prm=self.prm_head(h).squeeze(-1)
        a=self.attn(h).squeeze(-1).masked_fill(pad_mask, float("-inf"))
        w=torch.softmax(a,1).unsqueeze(-1); pooled=(h*w).sum(1)
        return recon, prm, self.chain_head(pooled).squeeze(-1), pooled

In [ ]:
import json, numpy as np, torch
recs = torch.load(f"{INP}/step_cache.pt", weights_only=False)
meta = {}
for l in open(f"{INP}/candidates.jsonl"):
    if l.strip():
        r=json.loads(l); meta[r["record_id"]]=r
model = StepSDAE_PRM(); model.load_state_dict(torch.load(f"{INP}/sdae_best.pt", map_location="cpu")); model.eval()
y = np.array([r["chain"] for r in recs])
print(f"{len(recs)} candidates | {int((y=='B').sum())} Type-B | base rate {(y=='B').mean():.3f}")

# confound targets
L,NS,LA,DS=[],[],[],[]
for r in recs:
    m=meta.get(r["id"],{}); t=m.get("response_text") or m.get("full_text") or ""
    L.append(np.log1p(len(t.split()))); NS.append(len(r["steps_text"]))
    LA.append((t.count(chr(92))+t.count("$"))/max(len(t.split()),1)); DS.append(r["split"])
L=np.array(L); NS=np.array(NS,float); LA=np.array(LA); DS=np.array(DS)

In [ ]:
# M1a — extract the three stages per candidate
H,Z,C=[],[],[]
with torch.no_grad():
    for r in recs:
        X=torch.from_numpy(r["steps_emb"]).float().unsqueeze(0)         # (1,T,384)
        pad=torch.zeros(1,X.size(1),dtype=torch.bool)
        h=model.encode(X,pad)                                            # (1,T,256) bottleneck codes
        _,_,_,pooled=model(X,pad,None)                                   # (1,256) attention-pooled
        H.append(r["steps_emb"].mean(0)); Z.append(h[0].mean(0).numpy()); C.append(pooled[0].numpy())
H=np.array(H); Z=np.array(Z); C=np.array(C)
print("H",H.shape,"Z",Z.shape,"c",C.shape)

In [ ]:
# M1b — at which stage does each confound become linearly available?
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import cross_val_score
stages={"H (per-step MiniLM, pooled)":H, "Z (bottleneck codes, pooled)":Z, "c (attention-pooled chain)":C}
cont={"log_length":L, "n_steps":NS, "latex_density":LA}
rows=[]
print(f"{'stage':<32}{'log_len R2':>12}{'n_steps R2':>12}{'latex R2':>11}{'dataset acc':>13}")
for sn,S in stages.items():
    r2=[cross_val_score(Ridge(1.0),S,v,cv=5,scoring='r2').mean() for v in cont.values()]
    acc=cross_val_score(LogisticRegression(max_iter=2000),S,DS,cv=5,scoring='accuracy').mean()
    rows.append((sn,*r2,acc))
    print(f"{sn:<32}{r2[0]:>12.3f}{r2[1]:>12.3f}{r2[2]:>11.3f}{acc:>13.3f}")
print("\nRead: if log_length/n_steps jump from H->Z, AGGREGATION manufactures the confound")
print("(MiniLM encodes steps independently, so it can't see total length/step-count).")

In [ ]:
# M1c — CAUSAL: project the length direction out of c; does it reproduce residualization?
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
# build the 4-confound matrix for residualization (matches the paper's protocol)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
surf=StandardScaler().fit_transform(np.c_[L,LA,NS])
oh=OneHotEncoder(sparse_output=False,handle_unknown='ignore').fit_transform(DS.reshape(-1,1))
Cf=np.hstack([np.ones((len(recs),1)),surf,oh])

def split(n,seed=0):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); cut=int(.8*n); return idx[:cut],idx[cut:]
def probe_f1(Zr,seed=0):
    tr,va=split(len(Zr),seed)
    clf=LogisticRegression(max_iter=2000).fit(Zr[tr],y[tr])
    return f1_score(y[va],clf.predict(Zr[va]),pos_label='B')

tr,_=split(len(C))
# (1) raw c ; (2) full residualization (paper's ~0.497) ; (3) project out length direction only
f1_raw=probe_f1(C)
beta=np.linalg.lstsq(Cf[tr],C[tr],rcond=None)[0]; f1_resid=probe_f1(C-Cf@beta)
w=Ridge(1.0).fit(C[tr],L[tr]).coef_; w=w/np.linalg.norm(w)
C_abl=C-np.outer(C@w,w)  # remove the length direction
f1_lenabl=probe_f1(C_abl)
print(f"A/B f1_B on c:")
print(f"  raw (no control)              {f1_raw:.3f}")
print(f"  full 4-confound residualized  {f1_resid:.3f}   <- paper's number")
print(f"  length-direction ablated only {f1_lenabl:.3f}   <- causal single-direction edit")
print("\nIf length-ablated approaches residualized, ONE direction (length) carries most of the")
print("confound: statistical control and a targeted causal edit agree. Mechanistic version of the protocol.")

In [ ]:
# Attention analysis — what does the chain head attend to: long steps (surface) or error steps (signal)?
pos_r, len_r = [], []
att_err, att_ok = [], []
with torch.no_grad():
    for r in recs:
        T=len(r["steps_text"])
        if T<2: continue
        X=torch.from_numpy(r["steps_emb"]).float().unsqueeze(0); pad=torch.zeros(1,T,dtype=torch.bool)
        h=model.encode(X,pad); a=model.attn(h).squeeze(-1)
        wts=torch.softmax(a,1)[0].numpy()
        steplen=np.array([len(s.split()) for s in r["steps_text"]],float)
        posn=np.linspace(0,1,T)
        if steplen.std()>0: len_r.append(np.corrcoef(wts,steplen)[0,1])
        pos_r.append(np.corrcoef(wts,posn)[0,1])
        lab=np.array(r["step_labels"])
        att_err += list(wts[lab==1]); att_ok += list(wts[(lab==0)])
print(f"chain-head attention weight correlations (mean over candidates):")
print(f"  vs step position : {np.nanmean(pos_r):+.3f}")
print(f"  vs step length   : {np.nanmean(len_r):+.3f}   (high +ve = attends to LONG steps = surface)")
print(f"  mean attn on ERROR steps  : {np.mean(att_err):.4f}")
print(f"  mean attn on non-error    : {np.mean(att_ok):.4f}")
print("  -> if error>>non-error, the head attends to the SIGNAL; if length corr is high, to SURFACE.")

In [ ]:
# Figure: 'where does difficulty live' — R2 by stage
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
labels=[r[0].split(' ')[0] for r in rows]
plt.figure(figsize=(7,4))
for i,name in enumerate(['log_length','n_steps','latex_density']):
    plt.plot(labels,[r[1+i] for r in rows],marker='o',label=name)
plt.ylabel('probe R² (5-fold)'); plt.xlabel('pipeline stage'); plt.title('Where the difficulty confound becomes linear')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig('m1_where_confound.png',dpi=140)
print('saved m1_where_confound.png'); plt.show()